In [7]:
import sys
from pathlib import Path
import pandas as pd

# Add current directory to path for importing scripts
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'scripts'))

df = pd.read_excel("../input/AugJobs_WithRunDashInName.xlsx")

# Filter to upstream and integrative only
df = df[df['jobname'].str.contains('upstream|integrative', case=False, na=False)].copy()

# Filter to only the jobs that completed
snap = df[df['stat'] == 'DONE']

# Select required columns
snap = snap[['jobid', 'jobname', 'run_time', 'mem_used', 'max_memory', 'core_eff']].copy()

# Convert memory from KB to GB
snap['mem_used_gb'] = snap['mem_used'] / (1024 ** 2)
snap['max_memory_gb'] = snap['max_memory'] / (1024 ** 2)
snap['run_time_hr'] = snap['run_time'] / 3600

# Drop raw columns
snap = snap.drop(columns=['mem_used', 'max_memory', 'run_time'])

In [8]:
stats = snap.groupby('jobname').agg(
    total_jobs=('jobid', 'count'),
    avg_run_time_hr=('run_time_hr', 'mean'),
    avg_mem_used_gb=('mem_used_gb', 'mean'),
    max_mem_used_gb=('mem_used_gb', 'max'),
    avg_max_memory_gb=('max_memory_gb', 'mean'),
    max_max_memory_gb=('max_memory_gb', 'max'),
    avg_core_eff=('core_eff', 'mean')
).round(2)
stats

,total_jobs,avg_run_time_hr,avg_mem_used_gb,max_mem_used_gb,avg_max_memory_gb,max_max_memory_gb,avg_core_eff
jobname,,,,,,,
run-integrative-analysis,12,0.87,90.88,140.67,106.55,144.11,16.26
run-upstream-analysis,16,17.17,111.44,414.51,575.37,2335.88,9.55
